<a href="https://colab.research.google.com/github/Vishali-Vivek006/Engineering-employability-prediction/blob/main/notebooks/04_modeltraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import joblib

from scipy import sparse

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [3]:


# Use NPZ if Phase 3 saved sparse matrices.
if "X_train_processed.npz" in uploaded:
    X_train = sparse.load_npz("X_train_processed.npz")
    X_test = sparse.load_npz("X_test_processed.npz")
else:
    X_train = np.load("X_train_processed.npy")
    X_test = np.load("X_test_processed.npy")

y_train = pd.read_csv("y_train.csv").squeeze().astype(int)
y_test = pd.read_csv("y_test.csv").squeeze().astype(int)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)
assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_test)).issubset({0, 1})

Training data: (4000, 169)
Testing data: (1000, 169)
Training labels: (4000,)
Testing labels: (1000,)


In [4]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        max_features=1.0,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        max_features=1.0,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    )
}

In [5]:
results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_accuracy = accuracy_score(y_train, train_pred)
    test_accuracy = accuracy_score(y_test, test_pred)

    precision = precision_score(
        y_test, test_pred, zero_division=0
    )
    recall = recall_score(
        y_test, test_pred, zero_division=0
    )
    f1 = f1_score(
        y_test, test_pred, zero_division=0
    )

    # ROC-AUC uses probabilities for the positive class (placed = 1).
    test_prob = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, test_prob)

    results.append({
        "Model": name,
        "Train_Accuracy": round(train_accuracy * 100, 2),
        "Test_Accuracy": round(test_accuracy * 100, 2),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1_Score": round(f1, 4),
        "ROC_AUC": round(roc_auc, 4),
        "Train_Test_Gap": round(
            (train_accuracy - test_accuracy) * 100, 2
        )
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(
    by="F1_Score", ascending=False
)

display(results_df)


Training Logistic Regression...


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Training Random Forest...

Training Extra Trees...

Training Gradient Boosting...


,Model,Train_Accuracy,Test_Accuracy,Precision,Recall,F1_Score,ROC_AUC,Train_Test_Gap
0,Logistic Regression,70.95,68.3,0.6177,0.6714,0.6434,0.7227,2.65
1,Random Forest,84.92,67.3,0.6148,0.6221,0.6184,0.7128,17.62
2,Extra Trees,82.90,65.6,0.5928,0.6150,0.6037,0.7087,17.30
3,Gradient Boosting,71.25,67.1,0.6474,0.5000,0.5642,0.7336,4.15


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy import sparse

# Scale processed features without centering sparse matrices
lr_model = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(
        max_iter=10000,
        class_weight="balanced",
        random_state=42,
        solver="lbfgs"
    )
)

lr_model.fit(X_train, y_train)

train_pred = lr_model.predict(X_train)
test_pred = lr_model.predict(X_test)
test_prob = lr_model.predict_proba(X_test)[:, 1]

print("Training Accuracy:", round(accuracy_score(y_train, train_pred) * 100, 2), "%")
print("Testing Accuracy :", round(accuracy_score(y_test, test_pred) * 100, 2), "%")
print("Test F1-score    :", round(f1_score(y_test, test_pred), 4))
print("Test ROC-AUC     :", round(roc_auc_score(y_test, test_prob), 4))

Training Accuracy: 71.03 %
Testing Accuracy : 68.2 %
Test F1-score    : 0.6435
Test ROC-AUC     : 0.7217


In [7]:

# ============================================================
# PHASE 4: SAVE THE PROVISIONAL MODEL
# ============================================================

import os
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 1. Check that required files and variables exist
if not os.path.exists("preprocessor.joblib"):
    raise FileNotFoundError(
        "preprocessor.joblib is missing. Upload your Phase 3 "
        "preprocessor.joblib before continuing."
    )

for variable in ["X_train", "X_test", "y_train", "y_test"]:
    if variable not in globals():
        raise NameError(
            f"{variable} is missing. Run the Phase 4 data-loading cell first."
        )

# 2. Build the provisional model
final_model = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(
        max_iter=10000,
        class_weight="balanced",
        random_state=42,
        solver="lbfgs"
    )
)

# 3. Train using training data only
final_model.fit(X_train, y_train)

# 4. Evaluate on the existing held-out test set
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print("PROVISIONAL MODEL: LOGISTIC REGRESSION")
print("-" * 45)
print(f"Training Accuracy: {final_model.score(X_train, y_train) * 100:.2f}%")
print(f"Testing Accuracy : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Precision        : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall           : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-score         : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC          : {roc_auc_score(y_test, y_prob):.4f}")

# 5. Save the fitted model and preserve the preprocessor
joblib.dump(final_model, "employability_model.joblib")

# Verify that both artifacts exist
assert os.path.exists("employability_model.joblib")
assert os.path.exists("preprocessor.joblib")

print("\nSaved successfully:")
print("1. employability_model.joblib")
print("2. preprocessor.joblib (preserved)")
print("\nModel file size:",
      round(os.path.getsize("employability_model.joblib") / 1024, 2),
      "KB")


PROVISIONAL MODEL: LOGISTIC REGRESSION
---------------------------------------------
Training Accuracy: 71.03%
Testing Accuracy : 68.20%
Precision        : 0.6159
Recall           : 0.6737
F1-score         : 0.6435
ROC-AUC          : 0.7217

Saved successfully:
1. employability_model.joblib
2. preprocessor.joblib (preserved)

Model file size: 6.61 KB


In [8]:

from google.colab import files

files.download("employability_model.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:

files.download("preprocessor.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>